# UMLS CHV (Consumer Health Vocabulary) Downloader

Notebook ini mengunduh data CHV dari UMLS REST API menggunakan API Key.

## Setup
Sebelum menjalankan notebook ini, buat file `.env` di folder yang sama dengan isi:
```
UMLS_API_KEY=your_api_key_here
```

**Jangan pernah hardcode API key langsung di kode!**

## Cell 1 — Import & Load API Key

In [ ]:
import os
import json
import time
import requests
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from tqdm.notebook import tqdm

# Load .env file
load_dotenv()
API_KEY = os.getenv('UMLS_API_KEY')

if not API_KEY:
    raise ValueError(
        "UMLS_API_KEY tidak ditemukan! "
        "Pastikan file .env sudah dibuat dengan isi: UMLS_API_KEY=your_key_here"
    )

print(f"API Key loaded: {'*' * (len(API_KEY) - 4)}{API_KEY[-4:]}")
print("Setup berhasil!")

## Cell 2 — Konfigurasi

In [ ]:
# ── Konfigurasi ──────────────────────────────────────────────────────────────
BASE_URL    = "https://uts-ws.nlm.nih.gov/rest"
VERSION     = "current"          # Gunakan versi terbaru
SOURCE      = "CHV"              # Consumer Health Vocabulary
OUTPUT_DIR  = Path("./data/chv")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Rate limiting — jangan terlalu cepat hit API
REQUEST_DELAY = 0.3              # detik antar request
PAGE_SIZE     = 25               # jumlah hasil per halaman (max 25 di UMLS API)

print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Konfigurasi siap.")

## Cell 3 — Helper Functions

In [ ]:
def umls_get(endpoint: str, params: dict = None) -> dict | None:
    """
    Helper untuk GET request ke UMLS REST API.
    API key otomatis disertakan di setiap request.
    Return None jika request gagal.
    """
    if params is None:
        params = {}
    
    # Sertakan API key di setiap request
    params["apiKey"] = API_KEY
    
    url = f"{BASE_URL}/{endpoint}"
    
    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.HTTPError as e:
        print(f"HTTP Error {response.status_code}: {e}")
        return None
    except requests.exceptions.ConnectionError:
        print("Connection error — periksa koneksi internet")
        return None
    except requests.exceptions.Timeout:
        print("Request timeout — coba lagi")
        return None
    except Exception as e:
        print(f"Error tidak terduga: {e}")
        return None


def validate_api_key() -> bool:
    """Validasi API key dengan hit endpoint sederhana."""
    result = umls_get(f"content/{VERSION}/CUI/C0009044")  # Test: 'Pleurisy'
    if result and "result" in result:
        print(f"✓ API Key valid! Test concept: {result['result']['name']}")
        return True
    else:
        print("✗ API Key tidak valid atau request gagal")
        return False


print("Helper functions loaded.")

## Cell 4 — Validasi API Key

In [ ]:
# Pastikan API key valid sebelum lanjut
is_valid = validate_api_key()

if not is_valid:
    raise Exception("Hentikan eksekusi — API key tidak valid!")

## Cell 5 — Search & Download CHV Concepts

Strategi download CHV:
1. Search UMLS dengan filter `sabs=CHV` untuk mendapatkan semua konsep CHV
2. Untuk setiap CUI yang ditemukan, ambil atoms CHV-nya (istilah awam)
3. Simpan ke CSV

In [ ]:
def search_chv_concepts(search_term: str, max_results: int = 100) -> list[dict]:
    """
    Search UMLS untuk konsep yang memiliki CHV term.
    Return list of {cui, name, chv_term}.
    """
    results = []
    page = 1

    while len(results) < max_results:
        data = umls_get(
            f"search/{VERSION}",
            params={
                "string"      : search_term,
                "sabs"        : SOURCE,       # filter ke CHV saja
                "returnIdType": "concept",
                "pageSize"    : PAGE_SIZE,
                "pageNumber"  : page
            }
        )

        if not data:
            break

        items = data.get("result", {}).get("results", [])
        if not items or items[0].get("ui") == "NONE":
            break  # Tidak ada hasil lagi

        for item in items:
            results.append({
                "cui" : item.get("ui"),
                "name": item.get("name"),
            })

        page += 1
        time.sleep(REQUEST_DELAY)

        if len(items) < PAGE_SIZE:
            break  # Halaman terakhir

    return results[:max_results]


def get_chv_atoms_for_cui(cui: str) -> list[dict]:
    """
    Ambil semua CHV atoms (istilah awam) untuk satu CUI.
    Setiap atom CHV berisi istilah yang dipahami pasien.
    """
    data = umls_get(
        f"content/{VERSION}/CUI/{cui}/atoms",
        params={
            "sabs"    : SOURCE,
            "pageSize": 25
        }
    )

    if not data:
        return []

    atoms = []
    for atom in data.get("result", []):
        atoms.append({
            "cui"         : cui,
            "atom_ui"     : atom.get("ui"),
            "chv_term"    : atom.get("name"),
            "source"      : atom.get("rootSource"),
            "term_type"   : atom.get("termType"),
            "language"    : atom.get("language"),
            "is_preferred": atom.get("preferred"),
        })

    return atoms


def get_concept_definition(cui: str) -> str:
    """
    Ambil definisi konsep dari CHV jika tersedia,
    fallback ke sumber lain jika tidak ada.
    """
    data = umls_get(
        f"content/{VERSION}/CUI/{cui}/definitions",
        params={"sabs": SOURCE}
    )

    if data and data.get("result"):
        return data["result"][0].get("value", "")
    return ""


print("Search & download functions loaded.")

## Cell 6 — Download CHV Data untuk Radiology Terms

Download CHV terms yang relevan untuk laporan radiologi toraks.
Query terms diambil dari terminologi umum chest X-ray reports.

In [ ]:
# Daftar query terms untuk radiologi toraks
# Ini adalah terminologi yang paling sering muncul di laporan CXR
RADIOLOGY_QUERY_TERMS = [
    # Temuan umum CXR
    "pleural effusion",
    "pneumonia",
    "consolidation",
    "opacity",
    "atelectasis",
    "pneumothorax",
    "cardiomegaly",
    "pulmonary edema",
    "infiltrate",
    "nodule",
    "mass",
    "effusion",
    "edema",
    "fibrosis",
    "emphysema",
    "bronchitis",
    "calcification",
    "carcinoma",
    "hemothorax",
    "mediastinum",
    # Anatomi
    "lung",
    "heart",
    "trachea",
    "diaphragm",
    "rib",
    "clavicle",
    "aorta",
    "bronchus",
    "hilum",
    "lobe",
    # Deskriptor
    "bilateral",
    "unilateral",
    "interstitial",
    "perihilar",
    "costophrenic angle",
    "hyperinflation",
    "hyperlucency",
    "haziness",
    "blunting",
    "silhouette sign",
]

print(f"Total query terms: {len(RADIOLOGY_QUERY_TERMS)}")
print("Siap untuk download.")

## Cell 7 — Jalankan Download

In [ ]:
all_records = []
failed_terms = []

print(f"Memulai download CHV data untuk {len(RADIOLOGY_QUERY_TERMS)} terms...\n")

for term in tqdm(RADIOLOGY_QUERY_TERMS, desc="Downloading CHV"):
    try:
        # Step 1: cari CUI yang punya CHV atom untuk term ini
        concepts = search_chv_concepts(term, max_results=5)

        if not concepts:
            failed_terms.append(term)
            continue

        for concept in concepts:
            cui = concept["cui"]

            # Step 2: ambil CHV atoms (istilah awam)
            atoms = get_chv_atoms_for_cui(cui)
            time.sleep(REQUEST_DELAY)

            # Step 3: ambil definisi (opsional, skip jika tidak ada)
            definition = get_concept_definition(cui)
            time.sleep(REQUEST_DELAY)

            for atom in atoms:
                all_records.append({
                    "query_term"    : term,
                    "cui"           : cui,
                    "medical_term"  : concept["name"],
                    "chv_lay_term"  : atom["chv_term"],
                    "atom_ui"       : atom["atom_ui"],
                    "term_type"     : atom["term_type"],
                    "is_preferred"  : atom["is_preferred"],
                    "definition"    : definition,
                    "language"      : atom["language"],
                    "source"        : atom["source"],
                })

    except Exception as e:
        print(f"\nError pada term '{term}': {e}")
        failed_terms.append(term)
        continue

print(f"\n✓ Selesai!")
print(f"  Total records: {len(all_records)}")
print(f"  Failed terms : {len(failed_terms)}")
if failed_terms:
    print(f"  Terms gagal  : {failed_terms}")

## Cell 8 — Simpan ke CSV & Preview

In [ ]:
if not all_records:
    print("Tidak ada data untuk disimpan.")
else:
    df = pd.DataFrame(all_records)

    # Hapus duplikat berdasarkan CUI + CHV lay term
    df_clean = df.drop_duplicates(subset=["cui", "chv_lay_term"])

    # Simpan full dataset
    output_path = OUTPUT_DIR / "chv_radiology_terms.csv"
    df_clean.to_csv(output_path, index=False, encoding="utf-8")

    # Simpan juga versi ringkas (hanya kolom paling penting)
    df_compact = df_clean[[
        "cui", "medical_term", "chv_lay_term",
        "is_preferred", "definition"
    ]].copy()
    compact_path = OUTPUT_DIR / "chv_radiology_compact.csv"
    df_compact.to_csv(compact_path, index=False, encoding="utf-8")

    print(f"✓ Disimpan ke: {output_path}")
    print(f"✓ Versi ringkas: {compact_path}")
    print(f"\nShape: {df_clean.shape}")
    print(f"Unique CUIs: {df_clean['cui'].nunique()}")
    print(f"Unique medical terms: {df_clean['medical_term'].nunique()}")
    print()
    display(df_clean.head(10))

## Cell 9 — Bonus: Download Bulk via UMLS Full Release (Alternatif)

Jika ingin download **seluruh** CHV dataset (bukan per-query), 
gunakan UMLS release file download. Ini lebih efisien untuk dataset besar.

> **Catatan:** File UMLS full release berukuran besar (~5GB). 
> CHV ada di dalam file `MRCONSO.RRF` dengan filter `SAB=CHV`.

In [ ]:
import zipfile
import shutil

def download_mrconso_chv_only(
    release_url: str = None,
    output_dir: Path = OUTPUT_DIR
) -> Path:
    """
    Download MRCONSO.RRF dari UMLS full release dan filter hanya baris CHV.
    Jauh lebih efisien daripada query API satu per satu.

    Args:
        release_url: URL UMLS release dari https://www.nlm.nih.gov/research/umls/licensedcontent/umlsknowledgesources.html
                     Contoh: https://download.nlm.nih.gov/umls/kss/2024AA/umls-2024AA-mrconso.zip
        output_dir : Folder output
    """
    if release_url is None:
        # Cari URL release terbaru
        releases = umls_get("releases/current")
        if releases and "result" in releases:
            print("Releases tersedia:")
            for rel in releases["result"][:3]:
                print(f"  - {rel}")
        print("\nSet release_url secara manual dengan URL dari halaman UMLS downloads.")
        return None

    # Download dengan API key
    download_url = f"{release_url}&apiKey={API_KEY}"
    zip_path = output_dir / "mrconso.zip"

    print(f"Downloading dari: {release_url}")
    print("Ini bisa memakan waktu lama (file besar)...")

    response = requests.get(download_url, stream=True)
    total = int(response.headers.get("content-length", 0))

    with open(zip_path, "wb") as f, tqdm(
        total=total, unit="iB", unit_scale=True, desc="Downloading"
    ) as bar:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            bar.update(len(chunk))

    print("Mengekstrak dan memfilter CHV...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(output_dir / "umls_extracted")

    # Filter MRCONSO.RRF untuk CHV saja
    mrconso = output_dir / "umls_extracted" / "MRCONSO.RRF"
    chv_out = output_dir / "chv_from_mrconso.tsv"

    # MRCONSO columns: CUI|LAT|TS|LUI|STT|SUI|ISPREF|AUI|SAUI|SCUI|SDUI|SAB|TTY|CODE|STR|SRL|SUPPRESS|CVF
    cols = ["CUI","LAT","TS","LUI","STT","SUI","ISPREF",
            "AUI","SAUI","SCUI","SDUI","SAB","TTY","CODE",
            "STR","SRL","SUPPRESS","CVF"]

    chv_rows = []
    with open(mrconso, encoding="utf-8", errors="ignore") as f:
        for line in tqdm(f, desc="Filtering CHV"):
            parts = line.strip().split("|")
            if len(parts) >= 12 and parts[11] == "CHV":
                chv_rows.append(dict(zip(cols, parts)))

    df_chv = pd.DataFrame(chv_rows)
    df_chv.to_csv(chv_out, index=False, sep="\t")

    # Cleanup zip
    zip_path.unlink()
    shutil.rmtree(output_dir / "umls_extracted")

    print(f"✓ CHV dari MRCONSO tersimpan: {chv_out}")
    print(f"  Total rows: {len(df_chv):,}")
    return chv_out


# Uncomment dan set URL untuk download full release:
# download_mrconso_chv_only(
#     release_url="https://download.nlm.nih.gov/umls/kss/2024AA/umls-2024AA-mrconso.zip"
# )

# Untuk sekarang, cek releases yang tersedia:
download_mrconso_chv_only()  # Tampilkan info releases tanpa download

## Cell 10 — Analisis & Statistik Data CHV

In [ ]:
# Load hasil download dari Cell 8
csv_path = OUTPUT_DIR / "chv_radiology_terms.csv"

if csv_path.exists():
    df_loaded = pd.read_csv(csv_path)

    print("=== Statistik Dataset CHV ===")
    print(f"Total rows          : {len(df_loaded):,}")
    print(f"Unique CUIs         : {df_loaded['cui'].nunique():,}")
    print(f"Unique medical terms: {df_loaded['medical_term'].nunique():,}")
    print(f"Unique CHV terms    : {df_loaded['chv_lay_term'].nunique():,}")
    print(f"Terms with definition: {df_loaded['definition'].notna().sum():,}")

    print("\n=== Sample Data ===")
    display(df_loaded[[
        "cui", "medical_term", "chv_lay_term",
        "is_preferred", "definition"
    ]].head(15))

    print("\n=== Top 10 Medical Terms dengan Paling Banyak CHV Lay Terms ===")
    top = (df_loaded.groupby("medical_term")["chv_lay_term"]
           .count().sort_values(ascending=False).head(10))
    display(top)

else:
    print(f"File tidak ditemukan: {csv_path}")
    print("Jalankan Cell 7 dan 8 terlebih dahulu.")

## Cell 11 — Export untuk KG Pipeline

Format output yang siap digunakan di pipeline KG-RAG.

In [ ]:
csv_path = OUTPUT_DIR / "chv_radiology_terms.csv"

if csv_path.exists():
    df_loaded = pd.read_csv(csv_path)

    # Format untuk KG pipeline:
    # Hanya ambil preferred lay term per CUI
    df_kg = (
        df_loaded[df_loaded["is_preferred"] == True]
        [["cui", "medical_term", "chv_lay_term", "definition"]]
        .drop_duplicates(subset=["cui"])
        .rename(columns={
            "medical_term" : "technical_term",
            "chv_lay_term" : "lay_term",
            "definition"   : "lay_definition"
        })
        .reset_index(drop=True)
    )

    # Simpan untuk KG pipeline
    kg_path = OUTPUT_DIR / "chv_for_kg_pipeline.csv"
    df_kg.to_csv(kg_path, index=False)

    # Simpan juga sebagai JSON (opsional)
    json_path = OUTPUT_DIR / "chv_for_kg_pipeline.json"
    df_kg.to_json(json_path, orient="records", indent=2)

    print(f"✓ KG-ready format disimpan: {kg_path}")
    print(f"✓ JSON format disimpan    : {json_path}")
    print(f"\nTotal unique CUI untuk KG : {len(df_kg):,}")
    print()
    display(df_kg.head(10))
else:
    print("Jalankan Cell 7-8 terlebih dahulu untuk menghasilkan data.")